In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
from src.features import *
from src.labeling import *
from src.volatility import *

In [2]:
df = pd.read_csv("../data/raw/nifty50.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

In [4]:
df = add_returns(df)
df = add_close_to_close_volatility(df, window=20)

df = add_target_direction(df)
df = add_lagged_returns(df, lags=(1,5,10))
df = add_moving_average_features(df, windows=(5,10,20))
df = add_volatility_features(df, vol_col="vol_cc", lags=(1,))

# UPDATED — use vol_cc for regimes
df = add_volatility_regime(df, vol_col="vol_cc")

df = df.dropna().reset_index(drop=True)
df.tail()

,Date,Open,High,Low,Close,Volume,return,vol_cc,target,ret_lag_1,ret_lag_5,ret_lag_10,ma_ratio_5,ma_ratio_10,ma_ratio_20,vol_cc_lag_1,vol_regime
4474,2026-02-13,25571.150391,25630.349609,25444.300781,25471.099609,453500,-0.013023,0.138191,1,-0.005650,0.001985,-0.003865,0.986987,0.991236,0.998418,0.130152,Medium
4475,2026-02-16,25423.599609,25697.000000,25372.699219,25682.750000,275800,0.008309,0.141515,1,-0.013023,0.006757,-0.009172,0.996614,0.997166,1.006737,0.138191,Medium
4476,2026-02-17,25637.949219,25764.400391,25570.300781,25725.400391,344100,0.001661,0.140728,1,0.008309,0.002623,0.025476,0.999897,0.998830,1.008133,0.141515,Medium
4477,2026-02-18,25752.650391,25828.050781,25645.150391,25819.349609,310200,0.003652,0.130728,0,0.001661,0.000721,0.001883,1.004599,1.002309,1.010652,0.140728,Medium
4478,2026-02-19,25873.349609,25885.300781,25388.750000,25454.349609,0,-0.014137,0.141139,0,0.003652,-0.005650,-0.005168,0.993124,0.988863,0.995786,0.130728,Medium


In [5]:
feature_cols = [
    "ret_lag_1",
    "ret_lag_5",
    "ret_lag_10",
    "ma_ratio_5",
    "ma_ratio_10",
    "ma_ratio_20",
    "vol_cc",
    "vol_cc_lag_1"
]

In [6]:
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

# **Random Forest and Gradient Boost training**

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(train_df[feature_cols])
y_train = train_df["target"]

X_test = scaler.transform(test_df[feature_cols])
y_test = test_df["target"]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

test_df["pred_rf"] = rf.predict(X_test)
test_df["pred_gb"] = gb.predict(X_test)

In [8]:
def regime_accuracy(df, pred_col):
    return (
        df.groupby("vol_regime")
        .apply(lambda g: (g[pred_col] == g["target"]).mean())
    )

reg_rf = regime_accuracy(test_df, "pred_rf")
reg_gb = regime_accuracy(test_df, "pred_gb")

regime_results = pd.DataFrame({
    "RandomForest": reg_rf,
    "GradientBoost": reg_gb
})

regime_results

,RandomForest,GradientBoost
vol_regime,,
High,0.658228,0.544304
Low,0.554007,0.587108
Medium,0.530864,0.510288


In [9]:
def walkforward_model(df, feature_cols, model):
    years = sorted(df["Date"].dt.year.unique())
    results = []

    for y in years:
        train = df[df["Date"].dt.year < y]
        test  = df[df["Date"].dt.year == y]

        if len(train) < 1000:
            continue

        X_train = train[feature_cols]
        y_train = train["target"]
        X_test  = test[feature_cols]
        y_test  = test["target"]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        acc = (pred == y_test).mean()

        results.append({"Year": y, "Accuracy": acc})

    return pd.DataFrame(results)

# **Walk-Forward Random Forest and Gradient Boost**

In [10]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

wf_rf = walkforward_model(df, feature_cols, rf_model)
wf_gb = walkforward_model(df, feature_cols, gb_model)

wf_rf.head(), wf_gb.head()

(   Year  Accuracy
 0  2012  0.495868
 1  2013  0.558704
 2  2014  0.566667
 3  2015  0.524590
 4  2016  0.495902,
    Year  Accuracy
 0  2012  0.500000
 1  2013  0.558704
 2  2014  0.529167
 3  2015  0.520492
 4  2016  0.479508)

In [11]:
wf_summary = pd.DataFrame({
    "Model": ["RandomForest", "GradientBoost"],
    "MeanAccuracy": [wf_rf["Accuracy"].mean(), wf_gb["Accuracy"].mean()],
    "Std": [wf_rf["Accuracy"].std(), wf_gb["Accuracy"].std()]
})

wf_summary

,Model,MeanAccuracy,Std
0,RandomForest,0.520786,0.044735
1,GradientBoost,0.524069,0.044422
